<a href="https://colab.research.google.com/github/Nipun1a/Credit_predict/blob/main/crc_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [239]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [240]:
# Loading the dataset to a pandas dataframe
credit_card_data = pd.read_csv('/content/credit_risk_dataset.csv')

In [241]:
#first 5 rows of the dataset
credit_card_data.head()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level,Housing_Status,Default
0,59,52154.0,11276,823,15,Bachelors,Own,0
1,49,116646.0,43663,315,5,PhD,Own,0
2,35,61157.0,18994,428,8,Masters,Own,1
3,63,52154.0,28499,408,26,Bachelors,Rent,0
4,28,148876.0,28040,832,3,Masters,Own,1


In [242]:
credit_card_data.tail()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level,Housing_Status,Default
995,53,44519.0,7307,433,22,PhD,Rent,1
996,22,107487.0,44901,582,7,High School,Own,1
997,34,102870.0,16205,372,29,Masters,Rent,0
998,60,66197.0,10906,780,24,PhD,Own,0
999,60,NaN,7591,651,5,High School,Mortgage,0


In [243]:
credit_card_data.shape

(1000, 8)

In [244]:
#dataset information
credit_card_data.info()

# The NameError for 'legit' indicates that the variable 'legit' has not been defined.
# Please ensure that the cell defining 'legit' and 'fraud' (cell RWxqh4OAT_B5) has been executed.
# The relevant code for defining 'legit' and 'fraud' is:
# legit = credit_card_data[credit_card_data.Class == 0]
# fraud = credit_card_data[credit_card_data.Class == 1]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               1000 non-null   int64  
 1   Income            985 non-null    float64
 2   Loan_Amount       1000 non-null   int64  
 3   Credit_Score      1000 non-null   int64  
 4   Employment_Years  1000 non-null   int64  
 5   Education_Level   1000 non-null   object 
 6   Housing_Status    1000 non-null   object 
 7   Default           1000 non-null   int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 62.6+ KB


In [245]:
# checking the number of missing values in each column
credit_card_data.isnull().sum()

# Fill missing 'Income' values with the mean of the column
credit_card_data['Income'] = credit_card_data['Income'].fillna(credit_card_data['Income'].mean())

In [246]:
# distribution of legit transaction & fraudulent transactions
credit_card_data['Default'].value_counts()

,count
Default,
0,862
1,138


This dataset is highly unbalanced

0 --> Normal Trandaction

1 --> fradulent Transaction

In [247]:
#seprating the data for analysis
legit = credit_card_data[credit_card_data.Default == 0]
fraud = credit_card_data[credit_card_data.Default == 1]

In [248]:
print(legit.shape)
print(fraud.shape)

(862, 8)
(138, 8)


In [249]:
# statistical measures of the data
legit.Loan_Amount.describe()

,Loan_Amount
count,862.000000
mean,27965.696056
std,12800.694225
min,5097.000000
25%,16537.250000
50%,28632.000000
75%,38926.000000
max,49976.000000


In [250]:
fraud.Loan_Amount.describe()

,Loan_Amount
count,138.000000
mean,26252.855072
std,12557.228545
min,5281.000000
25%,15882.500000
50%,25701.000000
75%,36128.000000
max,49298.000000


In [251]:
# compare the values for both transactions
credit_card_data.groupby('Default').mean(numeric_only=True)

,Age,Income,Loan_Amount,Credit_Score,Employment_Years
Default,,,,,
0,42.976798,82402.181639,27965.696056,580.453596,15.220418
1,39.630435,81328.747414,26252.855072,584.775362,14.811594


Under-Sampling

Build a sample dataset containing similar distribution of normal transactions and Fraudulent Transaction


Number of Fradulent Transactions --> 492

In [252]:
legit_sample = legit.sample(n=138)  # same as fraud

Concatenating Two dataFrames

In [253]:
new_dataset = pd.concat([legit_sample, fraud], axis = 0)

# One-Hot Encode categorical features for the new_dataset
new_dataset = pd.get_dummies(new_dataset, columns=['Education_Level', 'Housing_Status'], drop_first=True)

# Check for NaNs in new_dataset after all transformations
print("NaNs in new_dataset after one-hot encoding:\n", new_dataset.isnull().sum())

# Impute any remaining numeric NaNs in new_dataset, if any
for col in new_dataset.select_dtypes(include=np.number).columns:
    if new_dataset[col].isnull().any():
        new_dataset[col] = new_dataset[col].fillna(new_dataset[col].mean())

NaNs in new_dataset after one-hot encoding:
 Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Default                        0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64


In [254]:
new_dataset.head()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
309,41,116809.0,10249,848,21,0,False,True,False,False,False
302,50,52265.0,39503,561,9,0,False,False,False,False,True
728,64,122916.0,28225,847,6,0,False,True,False,False,False
550,45,52154.0,33082,465,5,0,False,False,False,True,False
608,28,52265.0,16889,783,12,0,True,False,False,False,True


In [255]:
new_dataset.tail()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
940,57,107487.0,34118,349,11,1,True,False,False,False,True
983,28,52265.0,5282,594,29,1,False,False,False,True,False
986,34,148876.0,9135,529,2,1,False,True,False,True,False
995,53,44519.0,7307,433,22,1,False,False,True,False,True
996,22,107487.0,44901,582,7,1,True,False,False,True,False


In [256]:
new_dataset['Default'].value_counts()

,count
Default,
0,138
1,138


In [257]:
new_dataset.groupby('Default').mean(numeric_only=True)

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
Default,,,,,,,,,,
0,43.275362,81406.464460,28967.413043,604.000000,15.811594,0.289855,0.268116,0.253623,0.340580,0.347826
1,39.630435,81328.747414,26252.855072,584.775362,14.811594,0.202899,0.282609,0.275362,0.326087,0.326087


Splitting the data into Features & Targets

In [258]:
X = new_dataset.drop(columns='Default', axis=1)
Y = new_dataset['Default']

In [259]:
print(X)

     Age    Income  Loan_Amount  Credit_Score  Employment_Years  \
309   41  116809.0        10249           848                21   
302   50   52265.0        39503           561                 9   
728   64  122916.0        28225           847                 6   
550   45   52154.0        33082           465                 5   
608   28   52265.0        16889           783                12   
..   ...       ...          ...           ...               ...   
940   57  107487.0        34118           349                11   
983   28   52265.0         5282           594                29   
986   34  148876.0         9135           529                 2   
995   53   44519.0         7307           433                22   
996   22  107487.0        44901           582                 7   

     Education_Level_High School  Education_Level_Masters  \
309                        False                     True   
302                        False                    False   
728         

In [260]:
print(Y)

309    0
302    0
728    0
550    0
608    0
      ..
940    1
983    1
986    1
995    1
996    1
Name: Default, Length: 276, dtype: int64


Split the data into Training data & Testing data

In [261]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [262]:
print(X.shape, X_train.shape, X_test.shape)

(276, 10) (220, 10) (56, 10)


Model Training

Logistic Regression

In [263]:
'''model = LogisticRegression(
    solver='liblinear',
    class_weight='balanced',
    max_iter=200,
    random_state=42
)'''
'''model = RandomForestClassifier(n_estimators=100, random_state=42)'''
from xgboost import XGBClassifier

model = XGBClassifier(scale_pos_weight=3)


In [264]:
# training the logistic Regression Model with Training Data
print("Checking for NaNs in X_train before fitting:")
print(X_train.isnull().sum())
print("Total NaNs in X_train:", X_train.isnull().sum().sum())
model.fit(X_train, Y_train)

Checking for NaNs in X_train before fitting:
Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64
Total NaNs in X_train: 0


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

Model Evaluation

Accuracy score

In [265]:
# accuracy on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [266]:
print('Accuracy on Training data:', training_data_accuracy)

Accuracy on Training data: 1.0


In [267]:
#acuracy on test data
print("Checking for NaNs in X_test before prediction:")
print(X_test.isnull().sum())
print("Total NaNs in X_test:", X_test.isnull().sum().sum())
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

Checking for NaNs in X_test before prediction:
Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64
Total NaNs in X_test: 0


In [268]:
print('Accuracy score on Test Data:', test_data_accuracy)

Accuracy score on Test Data: 0.5535714285714286


In [269]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       0.57      0.46      0.51        28
           1       0.55      0.64      0.59        28

    accuracy                           0.55        56
   macro avg       0.56      0.55      0.55        56
weighted avg       0.56      0.55      0.55        56



In [270]:
from sklearn.metrics import roc_auc_score

roc_auc_score(Y_test, model.predict_proba(X_test)[:,1])

np.float64(0.5012755102040816)

### Confusion Matrix

A confusion matrix is a table that is used to evaluate the performance of a classification model. It summarizes the number of correct and incorrect predictions made by the classifier with respect to the actual outcomes. It has four key components:

*   **True Positives (TP):** Correctly predicted positive values.
*   **True Negatives (TN):** Correctly predicted negative values.
*   **False Positives (FP):** Incorrectly predicted positive values (Type I error).
*   **False Negatives (FN):** Incorrectly predicted negative values (Type II error).

In [271]:
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
conf_matrix = confusion_matrix(Y_test, X_test_prediction)

print('Confusion Matrix:')
print(conf_matrix)

Confusion Matrix:
[[13 15]
 [10 18]]


In [272]:
Y_test.count()

np.int64(56)